# قراءة تجربة مختارات: دفتر مقدَّم

المختبر 1.6 · الذكاء الاصطناعي في التجزئة والاتصالات (USE-SEC-210).

هذا الدفتر مكمّل لجدول البيانات: يحسب الأرقام نفسها التي تحسبها في المختبر، ويضيف جدول الأسابيع الأربعة. لا تكتب فيه شيئًا؛ شغّل الخلايا بالترتيب وقارن.

1. في Colab أو JupyterLite ارفع الملف `ab_test_daily.csv` إلى المجلد نفسه (زر الرفع في شريط الملفات).
2. شغّل الخلايا. لا يحتاج الدفتر أي مكتبة خارج بايثون، ولا يتصل بأي نموذج.

In [ ]:
import csv, math
from pathlib import Path

CANDIDATES = ["ab_test_daily.csv", "../files/retail/ab_test_daily.csv", "files/retail/ab_test_daily.csv",
              "labs/files/retail/ab_test_daily.csv"]
path = next((Path(p) for p in CANDIDATES if Path(p).exists()), None)
if path is None:
    raise FileNotFoundError("Upload ab_test_daily.csv next to this notebook, then run this cell again.")
with path.open(encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "rows from", path)

## 1 · تطابق نسبة العينة

In [ ]:
def total(arm, col, weeks=(1, 2, 3, 4)):
    return sum(int(r[col]) for r in rows if r["arm"] == arm and int(r["week"]) in weeks)

nA, nB = total("A", "visitors"), total("B", "visitors")
expected = (nA + nB) / 2
chi2 = (nA - expected) ** 2 / expected + (nB - expected) ** 2 / expected
p = math.erfc(math.sqrt(chi2 / 2))
print(f"visitors A {nA:,}  B {nB:,}  share A {100 * nA / (nA + nB):.2f}%  chi2 {chi2:.2f}  p {p:.3f}")
print("sample ratio mismatch" if p < 0.001 else "no sample ratio mismatch")

## 2 · التحويل والإيراد لكل زائر، أسبوعًا أسبوعًا

In [ ]:
def readout(weeks):
    a = [total("A", c, weeks) for c in ("visitors", "orders", "revenue_sar", "returned_orders")]
    b = [total("B", c, weeks) for c in ("visitors", "orders", "revenue_sar", "returned_orders")]
    pa, pb = a[1] / a[0], b[1] / b[0]
    se = math.sqrt(pa * (1 - pa) / a[0] + pb * (1 - pb) / b[0])
    return dict(conv_a=100 * pa, conv_b=100 * pb, lift=100 * (pb / pa - 1),
                low=100 * (pb - pa - 1.96 * se), high=100 * (pb - pa + 1.96 * se),
                rpv_a=a[2] / a[0], rpv_b=b[2] / b[0], ret_a=100 * a[3] / a[1], ret_b=100 * b[3] / b[1])

print("period      conv A  conv B   lift%   95% CI of the difference (pts)   RPV A   RPV B  returns A  returns B")
for label, weeks in [("week 1", (1,)), ("week 2", (2,)), ("week 3", (3,)), ("week 4", (4,)),
                     ("weeks 3-4", (3, 4)), ("all", (1, 2, 3, 4))]:
    r = readout(weeks)
    print(f"{label:10s} {r['conv_a']:6.2f}  {r['conv_b']:6.2f}  {r['lift']:6.2f}   [{r['low']:6.3f}, {r['high']:6.3f}]"
          f"                  {r['rpv_a']:6.2f}  {r['rpv_b']:6.2f}  {r['ret_a']:8.2f}  {r['ret_b']:8.2f}")

In [ ]:
r = readout((1, 2, 3, 4))
print(f"revenue per visitor, whole test: A {r['rpv_a']:.2f} SAR, B {r['rpv_b']:.2f} SAR, change {100 * (r['rpv_b'] / r['rpv_a'] - 1):.2f}%")

## 3 · اقرأ

- هل يتغير الارتفاع من أسبوع إلى أسبوع؟ ماذا يسمى ذلك؟
- ما مقياس القرار المتفق عليه قبل التجربة، وماذا حدث له؟
- قارن كل رقم هنا بجدولك في المختبر.